# 02_analisis · Capa PLATA → análisis (A2.2)
**Grupo 03 · Retail / Comercio.** Responde las 3 preguntas de negocio con 5 consultas (Spark SQL y DataFrame API).
Incluye 1 agregación con función de ventana (Q2) y 1 join entre 2 fuentes (Q3).

| Pregunta | Consultas |
|---|---|
| P1 ¿Cuánto suben las ventas semanales en semanas festivas, por tipo de tienda? | Q1, Q2 |
| P2 ¿Dónde (tiendas, departamentos) se concentra la venta que hay que planear? | Q3 |
| P3 ¿Por qué varían las ventas: clima, combustible, economía o promociones? | Q4, Q5 |

> Las 3 preguntas son la propuesta del equipo derivada de la pregunta general de la Fase 1. Ajustarlas si el docente las pide distintas.

In [0]:
from pyspark.sql import functions as F, Window
ESQUEMA = "workspace.default"
ventas = spark.table(f"{ESQUEMA}.plata_ventas_enriquecida")
ventas.createOrReplaceTempView("v")
spark.table(f"{ESQUEMA}.plata_stores").createOrReplaceTempView("t")

## Q1 · Venta promedio por semana festiva y tipo de tienda (P1)

In [0]:
q1 = spark.sql("""
SELECT Type AS tipo_tienda, IsHoliday AS festivo,
       COUNT(*) AS registros,
       ROUND(AVG(Weekly_Sales), 2) AS venta_promedio,
       ROUND(PERCENTILE_APPROX(Weekly_Sales, 0.5), 2) AS venta_mediana
FROM v GROUP BY Type, IsHoliday ORDER BY Type, IsHoliday
""")
display(q1)

tipo_tienda,festivo,registros,venta_promedio,venta_mediana
A,false,199833,20055.0,10102.3
A,true,15160,21332.89,10871.95
B,false,151370,12202.53,6223.54
B,true,11449,13419.89,6505.91
C,false,39519,9546.08,1156.56
C,true,2954,9565.53,1240.7


**Interpretación Q1.** En semanas festivas la venta promedio sube ≈6,4 % en tiendas A (20.055 → 21.333) y ≈10,0 % en B (12.203 → 13.420); en C prácticamente no cambia (≈0,2 %). La mediana confirma la dirección (+7,6 % en A, +4,6 % en B, +7,3 % en C); en C el promedio casi no se mueve pero la mediana sí, lo que sugiere que la subida se da en los departamentos típicos y no en los de mayor volumen. _(Cifras de la corrida local con los mismos CSV; verificar contra la salida en Databricks.)_

## Q2 · Función de ventana: crecimiento semana a semana y ranking de semanas por tienda (P1)
Venta total semanal por tienda, variación frente a la semana anterior (`LAG`) y posición de cada semana dentro de su tienda (`RANK`).

In [0]:
q2 = spark.sql("""
WITH semanal AS (
  SELECT Store, Date, MAX(CAST(IsHoliday AS INT)) AS festivo, SUM(Weekly_Sales) AS venta_total
  FROM v GROUP BY Store, Date
)
SELECT Store, Date, festivo, ROUND(venta_total, 2) AS venta_total,
       ROUND(100 * (venta_total - LAG(venta_total) OVER (PARTITION BY Store ORDER BY Date))
             / LAG(venta_total) OVER (PARTITION BY Store ORDER BY Date), 2) AS var_pct_semana_previa,
       RANK() OVER (PARTITION BY Store ORDER BY venta_total DESC) AS ranking_en_tienda
FROM semanal
""")
q2.createOrReplaceTempView("q2")
display(spark.sql("SELECT festivo, ROUND(AVG(var_pct_semana_previa),2) AS var_prom_pct, COUNT(*) AS semanas FROM q2 GROUP BY festivo"))
display(spark.sql("SELECT Store, Date, festivo, venta_total FROM q2 WHERE ranking_en_tienda = 1 ORDER BY venta_total DESC LIMIT 10"))

festivo,var_prom_pct,semanas
0,0.58,5985
1,1.78,450


Store,Date,festivo,venta_total
14,2010-12-24,0,3818686.45
20,2010-12-24,0,3766687.43
10,2010-12-24,0,3749057.69
4,2011-12-23,0,3676388.98
13,2010-12-24,0,3595903.2
2,2010-12-24,0,3436154.68
27,2010-12-24,0,3078162.08
23,2010-12-24,0,2734277.1
6,2010-12-24,0,2727575.18
19,2010-12-24,0,2678206.42


**Interpretación Q2.** El crecimiento semanal promedio es 1,78 % en semanas marcadas como festivas frente a 0,58 % en las demás. Las diez semanas más altas del ranking son casi todas la del 24-dic-2010, que **no está marcada** como festiva en `IsHoliday` (la marca cae en la semana del 31-dic). Esto indica que la bandera del dataset subestima el pico real de Navidad y no debería usarse sola para planear inventario.

## Q3 · Join entre 2 fuentes: venta por tipo y tamaño de tienda, y venta por departamento (P2)
Une el resumen de ventas (fuente `train`) con el catálogo de tiendas (fuente `stores`).

In [0]:
q3 = spark.sql("""
SELECT t.Type AS tipo, COUNT(DISTINCT s.Store) AS tiendas,
       ROUND(SUM(s.venta), 0) AS venta_total,
       ROUND(SUM(s.venta) / SUM(t.Size) , 2) AS venta_por_unidad_de_tamano
FROM (SELECT Store, SUM(Weekly_Sales) AS venta FROM v GROUP BY Store) s
JOIN t ON s.Store = t.Store
GROUP BY t.Type ORDER BY venta_total DESC
""")
display(q3)
display(spark.sql("""
SELECT Dept, ROUND(SUM(Weekly_Sales),0) AS venta_total,
       ROUND(100*SUM(Weekly_Sales)/SUM(SUM(Weekly_Sales)) OVER (), 2) AS pct_del_total
FROM v GROUP BY Dept ORDER BY venta_total DESC LIMIT 10"""))

tipo,tiendas,venta_total,venta_por_unidad_de_tamano
A,22,4.331057479E9,1110.68
B,17,2.000741648E9,1163.06
C,6,4.05508022E8,1667.04


Dept,venta_total,pct_del_total
92,4.83943342E8,7.18
95,4.49320163E8,6.67
38,3.93118137E8,5.83
72,3.057282E8,4.54
90,2.91068464E8,4.32
40,2.88936022E8,4.29
2,2.80611174E8,4.17
91,2.16781706E8,3.22
13,1.9732157E8,2.93
8,1.94280781E8,2.88


**Interpretación Q3.** Las 22 tiendas A suman ≈4.331 M, las 17 B ≈2.001 M y las 6 C ≈405 M. Por unidad de tamaño las C rinden más (≈1.667 frente a ≈1.111 en A), es decir, venden más por unidad de tamaño. Los departamentos 92, 95 y 38 concentran ≈19,7 % de la venta total; el top-10 llega a ≈46 %. _(Verificar contra Databricks.)_

## Q4 · Correlación de la venta semanal con clima, combustible, CPI y desempleo (P3)

In [0]:
q4 = spark.sql("""
WITH s AS (
  SELECT Store, Date, SUM(Weekly_Sales) AS venta, AVG(Temperature) AS temp, AVG(Fuel_Price) AS comb,
         AVG(CPI) AS cpi, AVG(Unemployment) AS desempleo
  FROM v GROUP BY Store, Date)
SELECT ROUND(CORR(venta, temp),3) AS corr_temperatura, ROUND(CORR(venta, comb),3) AS corr_combustible,
       ROUND(CORR(venta, cpi),3) AS corr_cpi, ROUND(CORR(venta, desempleo),3) AS corr_desempleo
FROM s""")
display(q4)

corr_temperatura,corr_combustible,corr_cpi,corr_desempleo
-0.064,0.009,-0.073,-0.106


**Interpretación Q4.** Las correlaciones con la venta total por tienda-semana son débiles: temperatura −0,064, combustible 0,009, CPI −0,073 y desempleo −0,106. Mezclar tiendas de tamaños muy distintos oculta el efecto; una correlación baja aquí no descarta estas variables. Para la Fase 3 conviene medirla dentro de cada tienda o normalizar por tamaño.

## Q5 · Efecto de la intensidad de promoción × festivo desde noviembre de 2011 (P3)
Solo desde `2011-11-11`, la primera semana con MarkDown en el dataset. Desde ahí casi todas las tienda-semana tienen alguna
promoción, así que comparar "con" contra "sin" promoción es poco informativo; se compara por **intensidad**
(suma de MarkDown1-5: sin promoción / baja / alta, partiendo en la mediana de las semanas con promoción).

In [0]:
q5 = spark.sql("""
WITH base AS (
  SELECT Weekly_Sales, IsHoliday, MarkDown1+MarkDown2+MarkDown3+MarkDown4+MarkDown5 AS md_total
  FROM v WHERE Date >= DATE'2011-11-11'),
corte AS (SELECT PERCENTILE_APPROX(md_total, 0.5) AS mediana FROM base WHERE md_total > 0)
SELECT CASE WHEN md_total <= 0 THEN '1 sin promoción'
            WHEN md_total <= mediana THEN '2 promoción baja' ELSE '3 promoción alta' END AS intensidad,
       IsHoliday AS festivo, COUNT(*) AS registros, ROUND(AVG(Weekly_Sales), 2) AS venta_promedio
FROM base CROSS JOIN corte
GROUP BY 1, 2 ORDER BY 1, 2""")
display(q5)

intensidad,festivo,registros,venta_promedio
2 promoción baja,false,73119,13244.38
2 promoción baja,true,2399,9762.95
3 promoción alta,false,65877,19283.69
3 promoción alta,true,9553,19633.36


**Interpretación Q5.** Desde 11-nov-2011 no hay semanas sin promoción: todas tienen alguna. Con promoción alta la venta promedio es ≈19.300 frente a ≈13.200 con promoción baja en semanas normales. En festivo con promoción baja la venta cae a ≈9.760 (solo 2.399 registros), mientras que con promoción alta se mantiene ≈19.630. Es asociación, no causalidad: las promociones altas pueden concentrarse en tiendas grandes.